# Importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import nltk
from google.colab import drive
import pickle
import plotly.express as px

In [ ]:
# Para manter montada a pasta do Google Drive no COLAB
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


# Carregando os dados

## Trabalho

Carregando os dados de trabalho. Esses dados são informações de:
-

In [ ]:
path1 = '/content/drive/MyDrive/Dados_Empregos/'
arquivo1 = 'Combined_Jobs_Final.csv'

In [ ]:
empregos = pd.read_csv(path1 + arquivo1)
empregos.rename(columns={"Job.ID":"ID_emprego",
                         "Provider":"fornecedor",
                         "Status":"status",
                         "Slug":"slug",
                         "Title":"título",
                         "Position":"cargo",
                         "Company":"empresa",
                         "City":"cidade",
                         "State.Name":"nome_estado",
                         "State.Code":"código_estado",
                         "Address":"endereço",
                         "Latitude":"latitude",
                         "Longitude":"longitude",
                         "Industry":"indústria",
                         "Job.Description":"descrição_emprego",
                         "Requirements":"requisitos",
                         "Salary":"salário",
                         "Listing.Start":"início_listagem",
                         "Listing.End":"fim_listagem",
                         "Employment.Type":"tipo_emprego",
                         "Education.Required":"educação_requerida",
                         "Created.At":"criada_em",
                         "Updated.At":"atualizada_em"
                         }, inplace=True)
empregos.head(1)

,ID_emprego,fornecedor,status,slug,título,cargo,empresa,cidade,nome_estado,código_estado,...,indústria,descrição_emprego,requisitos,salário,início_listagem,fim_listagem,tipo_emprego,educação_requerida,criada_em,atualizada_em
0,111,1,open,palo-alto-ca-tacolicious-server,Server @ Tacolicious,Server,Tacolicious,Palo Alto,California,CA,...,Food and Beverages,Tacolicious' first Palo Alto store just opened...,NaN,8.0,NaN,NaN,Part-Time,NaN,2013-03-12 02:08:28 UTC,2014-08-16 15:35:36 UTC


Esses são dados sobre diversos empregos disponíveis no mercado. Temos informações sobre:
- ID_emprego: identifica o emprego
- status: se está disponível ou não
- título: é o cargo + a identidade da empresa
- cargo: cargo a ser ocupado
- empresa: nome da empresa contratante
- cidade: cidade em que o trabalho será feito
- nome_estado: estado onde o emprego é oferecido
- código_estado: identificação do estado em código
- indústria: tipo de indústria correlacionada à empresa
- descrição_emprego: descrição da vaga oferecida
- requisitos: se há ou não um requisito
- salário: quanto será pago por hora
- tipo_emprego: relacionado ao horário de trabalho, qual o emprego
- educação_requerida: se há ou não requisição de educação

In [ ]:
empregos.columns

Index(['ID_emprego', 'fornecedor', 'status', 'slug', 'título', 'cargo',
       'empresa', 'cidade', 'nome_estado', 'código_estado', 'endereço',
       'latitude', 'longitude', 'indústria', 'descrição_emprego', 'requisitos',
       'salário', 'início_listagem', 'fim_listagem', 'tipo_emprego',
       'educação_requerida', 'criada_em', 'atualizada_em'],
      dtype='object')

Temos alguns dados em falta

In [ ]:
empregos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84090 entries, 0 to 84089
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID_emprego          84090 non-null  int64  
 1   fornecedor          84090 non-null  int64  
 2   status              84090 non-null  object 
 3   slug                84090 non-null  object 
 4   título              84090 non-null  object 
 5   cargo               84090 non-null  object 
 6   empresa             81819 non-null  object 
 7   cidade              83955 non-null  object 
 8   nome_estado         83919 non-null  object 
 9   código_estado       83919 non-null  object 
 10  endereço            36 non-null     object 
 11  latitude            84090 non-null  float64
 12  longitude           84090 non-null  float64
 13  indústria           267 non-null    object 
 14  descrição_emprego   84034 non-null  object 
 15  requisitos          0 non-null      float64
 16  salá

#### Análise Exploratória dos Dados

Checando valores nulos

In [ ]:
empregos.isnull().sum() / len(empregos) * 100

,0
ID_emprego,0.000000
fornecedor,0.000000
status,0.000000
slug,0.000000
título,0.000000
cargo,0.000000
empresa,2.700678
cidade,0.160542
nome_estado,0.203354
código_estado,0.203354


Há muitos dados faltantes para requisitos, salário, indústria, endereço. Iremos desconsiderar essas colunas.

As restantes podemos inserir dados.

Vamos considerar apenas as colunas 'ID_emprego', 'título', 'cargo', 'empresa','cidade', 'tipo_emprego','descrição_emprego

In [ ]:
cols = ['ID_emprego']+['título']+['cargo']+ ['empresa']+['cidade']+['tipo_emprego']+['descrição_emprego']
empregos = empregos[cols]
empregos.columns = ['ID_emprego', 'título', 'cargo', 'empresa','cidade', 'tipo_emprego','descrição_emprego']
empregos.head()

,ID_emprego,título,cargo,empresa,cidade,tipo_emprego,descrição_emprego
0,111,Server @ Tacolicious,Server,Tacolicious,Palo Alto,Part-Time,Tacolicious' first Palo Alto store just opened...
1,113,Kitchen Staff/Chef @ Claude Lane,Kitchen Staff/Chef,Claude Lane,San Francisco,Part-Time,\r\n\r\nNew French Brasserie in S.F. Financia...
2,117,Bartender @ Machka Restaurants Corp.,Bartender,Machka Restaurants Corp.,San Francisco,Part-Time,We are a popular Mediterranean wine bar and re...
3,121,Server @ Teriyaki House,Server,Teriyaki House,Brisbane,Part-Time,● Serve food/drinks to customers in a profess...
4,127,Kitchen Staff/Chef @ Rosa Mexicano - Sunset,Kitchen Staff/Chef,Rosa Mexicano - Sunset,Los Angeles,Part-Time,"Located at the heart of Hollywood, we are one ..."


In [ ]:
# Checando novamente valores nulos
empregos.isnull().sum() / len(empregos) * 100

,0
ID_emprego,0.000000
título,0.000000
cargo,0.000000
empresa,2.700678
cidade,0.160542
tipo_emprego,0.011892
descrição_emprego,0.066595


Vamos tentar preencher esses dados vazios com as informações que temos disponíveis.

Coluna 'City'

In [ ]:
cidade_vazias = empregos[pd.isnull(empregos['cidade'])]
cidade_vazias.groupby(['empresa'])['cidade'].count()

,cidade
empresa,
Academic Year In America,0
CBS Healthcare Services and Staffing,0
CHI Payment Systems,0
Driveline Retail,0
Educational Testing Services,0
Genesis Health System,0
Genesis Health Systems,0
Home Instead Senior Care,0
St. Francis Hospital,0


Nós vemos que há 10 empresas que têm valores NaN na coluna cidade, então deve ser adicionado manualmente suas sedes (simplesmente pesquisando no google)

In [ ]:
#replacing nan with thier headquarters location
empregos['empresa'] = empregos['empresa'].replace(['Genesis Health Systems'], 'Genesis Health System')
empregos.loc[empregos.empresa == 'CHI Payment Systems', 'cidade'] = 'Illinois'
empregos.loc[empregos.empresa == 'Academic Year In America', 'cidade'] = 'Stamford'
empregos.loc[empregos.empresa == 'CBS Healthcare Services and Staffing ', 'cidade'] = 'Urbandale'
empregos.loc[empregos.empresa == 'Driveline Retail', 'cidade'] = 'Coppell'
empregos.loc[empregos.empresa == 'Educational Testing Services', 'cidade'] = 'New Jersey'
empregos.loc[empregos.empresa == 'Genesis Health System', 'cidade'] = 'Davennport'
empregos.loc[empregos.empresa == 'Home Instead Senior Care', 'cidade'] = 'Nebraska'
empregos.loc[empregos.empresa == 'St. Francis Hospital', 'cidade'] = 'New York'
empregos.loc[empregos.empresa == 'Volvo Group', 'cidade'] = 'Washington'
empregos.loc[empregos.empresa == 'CBS Healthcare Services and Staffing', 'cidade'] = 'Urbandale'

In [ ]:
empregos.isnull().sum()

ID_emprego              0
título                  0
cargo                   0
empresa              2271
cidade                  0
tipo_emprego           10
descrição_emprego      56
dtype: int64

Coluna 'tipo_emprego'

In [ ]:
empregos['tipo_emprego'].unique()

array(['Part-Time', 'Full-Time/Part-Time', 'Seasonal/Temp', 'Per Diem',
       'Intern', nan, 'Full-Time', 'Contract', 'Temporary/seasonal'],
      dtype=object)

In [ ]:
tipos_empregos_vazios = empregos[pd.isnull(empregos['tipo_emprego'])]
tipos_empregos_vazios.head()

,ID_emprego,título,cargo,empresa,cidade,tipo_emprego,descrição_emprego
10768,153197,Driving Partner @ Uber,Driving Partner,Uber,San Francisco,NaN,Uber is changing the way the world moves. From...
10769,153198,Driving Partner @ Uber,Driving Partner,Uber,Los Angeles,NaN,Uber is changing the way the world moves. From...
10770,153199,Driving Partner @ Uber,Driving Partner,Uber,Chicago,NaN,Uber is changing the way the world moves. From...
10771,153200,Driving Partner @ Uber,Driving Partner,Uber,Boston,NaN,Uber is changing the way the world moves. From...
10772,153201,Driving Partner @ Uber,Driving Partner,Uber,Ann Arbor,NaN,Uber is changing the way the world moves. From...


In [ ]:
tipos_empregos_vazios.groupby(['empresa'])['tipo_emprego'].size()

,tipo_emprego
empresa,
Uber,10


Os dados nulos são apenas para a empresa Uber, vamos considerar uma jornada de trabalho de período integral / meio período

In [ ]:
# Preenchendo valores nulos com meio período / período integral
empregos['tipo_emprego']=empregos['tipo_emprego'].fillna('Full-Time/Part-Time')

In [ ]:
empregos.isnull().sum()

,0
ID_emprego,0
título,0
cargo,0
empresa,2271
cidade,0
tipo_emprego,0
descrição_emprego,56


Vamos manter as informações de empresa e descrição nulas, pois combinaremos as colunas posição, empresa, cidade, tipo de emprego e posição. As que tiverem dados vazios serão considerados ''.

In [ ]:
empregos["texto"] = empregos["cargo"].map(str) + " " + empregos["empresa"] +" "+ empregos["cidade"]+ " "+empregos['tipo_emprego']+" "+empregos['descrição_emprego'] +" "+empregos['título']
empregos.head(2)

,ID_emprego,título,cargo,empresa,cidade,tipo_emprego,descrição_emprego,texto
0,111,Server @ Tacolicious,Server,Tacolicious,Palo Alto,Part-Time,Tacolicious' first Palo Alto store just opened...,Server Tacolicious Palo Alto Part-Time Tacolic...
1,113,Kitchen Staff/Chef @ Claude Lane,Kitchen Staff/Chef,Claude Lane,San Francisco,Part-Time,\r\n\r\nNew French Brasserie in S.F. Financia...,Kitchen Staff/Chef Claude Lane San Francisco P...


Vamos extrair apenas o ID, texto e título do trabalho

In [ ]:
todos_empregos = empregos[['ID_emprego', 'texto', 'título']]

todos_empregos = todos_empregos.fillna(" ")

todos_empregos.head()

,ID_emprego,texto,título
0,111,Server Tacolicious Palo Alto Part-Time Tacolic...,Server @ Tacolicious
1,113,Kitchen Staff/Chef Claude Lane San Francisco P...,Kitchen Staff/Chef @ Claude Lane
2,117,Bartender Machka Restaurants Corp. San Francis...,Bartender @ Machka Restaurants Corp.
3,121,Server Teriyaki House Brisbane Part-Time ● Se...,Server @ Teriyaki House
4,127,Kitchen Staff/Chef Rosa Mexicano - Sunset Los ...,Kitchen Staff/Chef @ Rosa Mexicano - Sunset


In [ ]:
todos_empregos.shape

(84090, 3)

Temos 84090 registros, sendo cda um um emprego, e 3 colunas

In [ ]:
todos_empregos.isnull().sum()

,0
ID_emprego,0
texto,0
título,0


Vamos baixar da biblioteca nltk os pacotes:
- punkt: responsável por tokenizar as palavras
- stopwords: responsável por remover as palavras que não agregam valor ao texto
- wordnet: responsável por lematizar as palavras
- averaged_perceptron_tagger: responsável por classificar as palavras em substantivos, verbos, adjetivos, etc

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

Vamos importar os pacotes baixados e criar duas funções:
- black_txt: responsável por remover as palavras que não agregam valor ao texto
- clean_txt: responsável por lematizar as palavras

In [ ]:
from nltk.corpus import stopwords
import re
import string
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize
from nltk.corpus import stopwords
stop = stopwords.words('english')
stop_words_ = set(stopwords.words('english'))
wn = WordNetLemmatizer()

def black_txt(token):
    return  token not in stop_words_ and token not in list(string.punctuation)  and len(token)>2

def clean_txt(text):
  clean_text = []
  clean_text2 = []
  text = re.sub("'", "",text)
  text=re.sub("(\\d|\\W)+"," ",text)
  text = text.replace("nbsp", "")
  clean_text = [ wn.lemmatize(word, pos="v") for word in word_tokenize(text.lower()) if black_txt(word)]
  clean_text2 = [word for word in clean_text if black_txt(word)]
  return " ".join(clean_text2)


Limpando o texto

Finalmente nós temos o texto limpo

In [ ]:
todos_empregos.head()

,ID_emprego,texto,título
0,111,Server Tacolicious Palo Alto Part-Time Tacolic...,Server @ Tacolicious
1,113,Kitchen Staff/Chef Claude Lane San Francisco P...,Kitchen Staff/Chef @ Claude Lane
2,117,Bartender Machka Restaurants Corp. San Francis...,Bartender @ Machka Restaurants Corp.
3,121,Server Teriyaki House Brisbane Part-Time ● Se...,Server @ Teriyaki House
4,127,Kitchen Staff/Chef Rosa Mexicano - Sunset Los ...,Kitchen Staff/Chef @ Rosa Mexicano - Sunset


Pronto, agora temos um dataframe com informações descritivas de cada emprego, sua identificação e nome do cargo e empresa.

In [ ]:
todos_empregos["ID_emprego"].nunique()

84090

### Dados usuários
Vamos pegar o dataset de trabalhos visualizados

#### Dados históricos dos usuários


In [ ]:
arquivo2 = 'Job_Views.csv'
historico_pesquisa = pd.read_csv(path1 + arquivo2)
historico_pesquisa.rename(columns={"Applicant.ID":"ID_candidato",
                                   "Job.ID":"ID_emprego",
                                   "Title":"título",
                                   "Position":"cargo",
                                   "Company":"empresa",
                                   "City":"cidade",
                                   "State.Name":"nome_estado",
                                   "Industry":"indústria",
                                   "View.Start":"início_visualização",
                                   "View.End":"fim_visualização",
                                   "View.Duration":"tempo_visualização",
                                   "Created.At":"criado_em",
                                   "Updated.At":"atualizado_em"
                                   }, inplace=True)
historico_pesquisa.head(1)

,ID_candidato,ID_emprego,título,cargo,empresa,cidade,nome_estado,State.Code,indústria,início_visualização,fim_visualização,tempo_visualização,criado_em,atualizado_em
0,10000,73666,Cashiers & Valets Needed! @ WallyPark,Cashiers & Valets Needed!,WallyPark,Newark,New Jersey,NJ,NaN,2014-12-12 20:12:35 UTC,2014-12-12 20:31:24 UTC,1129.0,2014-12-12 20:12:35 UTC,2014-12-12 20:12:35 UTC


Aqui temos informações sobre o histórico de pesquisa de cada usuário. Cada vaga visualizada pelo usuário nos trará informações sobre a identidade daquele usuário, o ID do emprego visto, o título do emprego, o cargo, a empresa, cidade, nome do estado, etc.

In [ ]:
historico_pesquisa.shape

(12370, 14)

In [ ]:
historico_pesquisa["ID_candidato"].nunique()

3448

Temos um total de 3448 candidatos

In [ ]:
historico_pesquisa["ID_emprego"].nunique()

7047

Há um total de 7047 empregos visualizados

Nesse caso usaremos apenas as colunas 'ID_candidato', 'ID_emprego', 'cargo', 'empresa','cidade'

In [ ]:
historico_pesquisa = historico_pesquisa[['ID_candidato', 'ID_emprego', 'cargo', 'empresa','cidade']]
historico_pesquisa["selecionar_cargo_empresa_cidade"] = historico_pesquisa["cargo"].map(str) + "  " + historico_pesquisa["empresa"] +"  "+ historico_pesquisa["cidade"]
historico_pesquisa["selecionar_cargo_empresa_cidade"] = historico_pesquisa['selecionar_cargo_empresa_cidade'].map(str).apply(clean_txt)
historico_pesquisa['selecionar_cargo_empresa_cidade'] = historico_pesquisa['selecionar_cargo_empresa_cidade'].str.lower()
historico_pesquisa = historico_pesquisa[['ID_candidato','selecionar_cargo_empresa_cidade']]
historico_pesquisa.head()


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
historico_pesquisa.shape

(12370, 2)

In [ ]:
historico_pesquisa["ID_candidato"].nunique()

3448

Pronto, agora temos um data frame de vagas de emprego visualizadas por usuários, contendo informações de cargos, empresas e cidades de interesse, além da identificação de cada candidato.

#### Dataset de experiência dos usuários


In [ ]:
arquivo3 = 'Experience.csv'
experiencia = pd.read_csv(path1 + arquivo3)
experiencia.rename(columns={"Applicant.ID":"ID_candidato",
                                   "Position.Name":"nome_cargo",
                                   "Employer.Name":"nome_chefe",
                                   "City":"cidade",
                                   "State.Name":"estado",
                                   "City":"cidade",
                                   "State.Name":"nome_estado",
                                   "State.Code":"código_estado",
                                   "Start.Date":"data_início",
                                   "End.Date":"data_fim",
                                   "Job.Description":"descrição_emprego",
                                   "Salary":"salário",
                                   "Can.Contact.Employer":"pode_contatar_chefe",
                                   "Created.At":"criado_em",
                                   "Updated.At":"atualizado_em"
                                   }, inplace=True)
experiencia.head(1)

,ID_candidato,nome_cargo,nome_chefe,cidade,nome_estado,código_estado,data_início,data_fim,descrição_emprego,salário,pode_contatar_chefe,criado_em,atualizado_em
0,10001,Account Manager / Sales Administration / Quali...,Barcode Resourcing,Bellingham,Washington,WA,2012-10-15,NaN,NaN,NaN,NaN,2014-12-12 20:10:02 UTC,2014-12-12 20:10:02 UTC


Esse dataset nos dá informações sobre a experiência de usuários. Cada um pode ter mais de uma experiência, por isso temos 8653 linhas e 3790 delas são usuários únicos. A experiência diz sobre o id do candidato, o nome do chefe, a cidade que trabalho e estado, data de início e fim desse emprego, salário, etc.

In [ ]:
experiencia.shape

(8653, 13)

In [ ]:
experiencia["ID_candidato"].nunique()

3790

In [ ]:
experiencia["ID_candidato"].nunique()

3790

Vamos aplicar a limpeza do texto dos dados

In [ ]:
experiencia= experiencia[['ID_candidato','nome_cargo']]
experiencia['nome_cargo'] = experiencia['nome_cargo'].map(str).apply(clean_txt)
experiencia.head()


,ID_candidato,nome_cargo
0,10001,account manager sales administration quality a...
1,10001,electronics technician item master controller
2,10001,machine operator
3,10003,maintenance technician
4,10003,electrical helper


In [ ]:
experiencia =  experiencia.sort_values(by='ID_candidato')
experiencia = experiencia.fillna(" ")
experiencia.head()


,ID_candidato,nome_cargo
2762,2,writer uloop blog
2763,2,volunteer
3759,3,market intern
3758,3,server
3757,3,prep cook


O mesmo candidato tem 3 candidaturas 100001 em uma única linha, então precisamos juntá-los

In [ ]:
experiencia = experiencia.groupby('ID_candidato', sort=False)['nome_cargo'].apply(' '.join).reset_index()
experiencia.head(5)

,ID_candidato,nome_cargo
0,2,writer uloop blog volunteer
1,3,market intern server prep cook
2,6,project assistant
3,8,deli clerk server cashier food prep order taker
4,11,cashier


Pronto, agora temos um data frame de experiência de cada usuário, com informações do nome do emprego e o cargo ocupado

#### Dataset de cargo de interesse

Aqui temos um data frame sobre cargos que o usuário declarou como cargo de interesse ao efetuar o cadastro ou procura na plataforma

In [ ]:
arquivo4 = 'Positions_Of_Interest.csv'
cargos_interesse = pd.read_csv(path1 + arquivo4)
cargos_interesse.rename(columns={"Applicant.ID":"ID_candidato",
                                 "Position.Of.Interest":"cargo_interesse",
                                 "Created.At":"criado_em",
                                 "Updated.At":"atualizado_em"}, inplace=True)
cargos_interesse = cargos_interesse.sort_values(by='ID_candidato')
cargos_interesse.head()

,ID_candidato,cargo_interesse,criado_em,atualizado_em
6437,96,Server,2014-08-14 15:56:42 UTC,2015-02-26 20:35:12 UTC
1158,153,Sales Rep,2014-08-14 15:56:47 UTC,2015-03-02 02:13:08 UTC
1155,153,Host,2014-08-14 15:56:42 UTC,2015-02-26 20:35:12 UTC
1154,153,Server,2014-08-14 15:56:42 UTC,2015-02-26 20:35:12 UTC
1156,153,Barista,2014-08-14 15:56:43 UTC,2015-02-18 02:35:06 UTC


In [ ]:
cargos_interesse.shape

(6560, 4)

Temos 6560 informações sobre cargos de interesse de usuários

In [ ]:
cargos_interesse["ID_candidato"].nunique()

2068

Temos 2068 usuários únicos, então grande parte tem mais de uma experiência

Vamos esquecer as colunas de "criado" e "atualizado em"

In [ ]:
cargos_interesse = cargos_interesse.drop(['criado_em', 'atualizado_em'], axis=1)

Vamos limpar a coluna cargo de interesse

In [ ]:
cargos_interesse['cargo_interesse']=cargos_interesse['cargo_interesse'].map(str).apply(clean_txt)
cargos_interesse = cargos_interesse.fillna(" ")
cargos_interesse.head(10)

,ID_candidato,cargo_interesse
6437,96,server
1158,153,sales rep
1155,153,host
1154,153,server
1156,153,barista
1157,153,customer service rep
1953,256,receptionist
1954,256,book keeper
1952,256,host
1951,256,server


Vamos unir todos os usuários, mostrando que cada um pode ter ou não mais de uma experiência

In [ ]:
cargos_interesse = cargos_interesse.groupby('ID_candidato', sort=True)['cargo_interesse'].apply(' '.join).reset_index()
cargos_interesse.head()

,ID_candidato,cargo_interesse
0,96,server
1,153,sales rep host server barista customer service...
2,256,receptionist book keeper host server sales rep...
3,438,host server customer service rep barista
4,568,book keeper customer service rep receptionist


####  Junção dos três

Vamos rever os datasets que fizemos

In [ ]:
todos_empregos.head()

,ID_emprego,texto,título
0,111,server tacolicious palo alto part time tacolic...,Server @ Tacolicious
1,113,kitchen staff chef claude lane san francisco p...,Kitchen Staff/Chef @ Claude Lane
2,117,bartender machka restaurants corp san francisc...,Bartender @ Machka Restaurants Corp.
3,121,server teriyaki house brisbane part time serve...,Server @ Teriyaki House
4,127,kitchen staff chef rosa mexicano sunset los an...,Kitchen Staff/Chef @ Rosa Mexicano - Sunset


In [ ]:
historico_pesquisa.head()

,ID_candidato,selecionar_cargo_empresa_cidade
0,10000,cashier valet need wallypark newark
1,10000,macys seasonal retail fragrance cashier garden...
2,10001,part time showroom sales cashier grizzly indus...
3,10002,event specialist part time advantage sales mar...
4,10002,bonefish kitchen staff bonefish grill greenville


In [ ]:
experiencia.head(5)

,ID_candidato,nome_cargo
0,2,writer uloop blog volunteer
1,3,market intern server prep cook
2,6,project assistant
3,8,deli clerk server cashier food prep order taker
4,11,cashier


In [ ]:
cargos_interesse.head()

,ID_candidato,cargo_interesse
0,96,server
1,153,sales rep host server barista customer service...
2,256,receptionist book keeper host server sales rep...
3,438,host server customer service rep barista
4,568,book keeper customer service rep receptionist


Mesclando os datasets de trabalhos e experiência

In [ ]:
experiencia_historico = historico_pesquisa.merge(experiencia, left_on='ID_candidato', right_on='ID_candidato', how='outer')
experiencia_historico = experiencia_historico.fillna(' ')
experiencia_historico = experiencia_historico.sort_values(by='ID_candidato')
experiencia_historico.head()

,ID_candidato,selecionar_cargo_empresa_cidade,nome_cargo
12370,2,,writer uloop blog volunteer
12371,3,,market intern server prep cook
12372,6,,project assistant
12373,8,,deli clerk server cashier food prep order taker
12374,11,,cashier


Mesclando o cargo de interesse com o dataframe existente

In [ ]:
experiencia_historico_interesse = experiencia_historico.merge(cargos_interesse, left_on='ID_candidato', right_on='ID_candidato', how='outer')
experiencia_historico_interesse = experiencia_historico_interesse.fillna(' ')
experiencia_historico_interesse = experiencia_historico_interesse.sort_values(by='ID_candidato')
experiencia_historico_interesse.head()

,ID_candidato,selecionar_cargo_empresa_cidade,nome_cargo,cargo_interesse
0,2,,writer uloop blog volunteer,
1,3,,market intern server prep cook,
2,6,,project assistant,
3,8,,deli clerk server cashier food prep order taker,
4,11,,cashier,


Combinando todas as colunas

In [ ]:
experiencia_historico_interesse["texto"] = experiencia_historico_interesse["selecionar_cargo_empresa_cidade"].map(str) + experiencia_historico_interesse["nome_cargo"] +" "+ experiencia_historico_interesse["cargo_interesse"]
experiencia_historico_interesse.head()

,ID_candidato,selecionar_cargo_empresa_cidade,nome_cargo,cargo_interesse,texto
0,2,,writer uloop blog volunteer,,writer uloop blog volunteer
1,3,,market intern server prep cook,,market intern server prep cook
2,6,,project assistant,,project assistant
3,8,,deli clerk server cashier food prep order taker,,deli clerk server cashier food prep order tak...
4,11,,cashier,,cashier


Selecionando apenas as colunas "Applicant.ID" e "text"

In [ ]:
df_candidato_final = experiencia_historico_interesse[['ID_candidato','texto']]
df_candidato_final.head()

,ID_candidato,texto
0,2,writer uloop blog volunteer
1,3,market intern server prep cook
2,6,project assistant
3,8,deli clerk server cashier food prep order tak...
4,11,cashier


In [ ]:
df_candidato_final['texto'] = df_candidato_final['texto'].apply(clean_txt)
df_candidato_final.head()

<ipython-input-144-d58c7dff40d2>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_candidato_final['texto'] = df_candidato_final['texto'].apply(clean_txt)


,ID_candidato,texto
0,2,writer uloop blog volunteer
1,3,market intern server prep cook
2,6,project assistant
3,8,deli clerk server cashier food prep order taker
4,11,cashier


Para testar o recomendador, selecionamos o usuário cujo ID é 326

In [ ]:
candidato = 326
indice = np.where(df_candidato_final['ID_candidato'] == candidato)[0][0]
user_q = df_candidato_final.iloc[[indice]]
user_q

,ID_candidato,texto
186,326,java developer


In [ ]:
df_candidato_final.shape

(15959, 2)

In [ ]:
df_candidato_final["ID_candidato"].nunique()

7037

In [ ]:
df_candidato_final["texto"].nunique()

10043

In [ ]:
df_candidato_final['ID_candidato'].values

array([    2,     3,     6, ..., 14639, 14642, 14643])

## Os sistemas de recomendação


#### TF-IDF

#### Count Vectorizer

####  Similaridade do Cosseno

É a métrica mais usada para calcular a semelhança entre textos. Matematizacente ela mede o cosseno do ângulo entre dois vetores projetados em um espaço multidimensional.

![image.png](attachment:image.png)

Vemos aqui que o emprego de Engenheiro de Machine Learning e Cientista de Dados possuem uma similaridade, seu $ \theta $ é próximo de 0, o que fará seu cosseno ser próximo de 1.

![image.png](attachment:image.png)

Agora comparando dois empregos diferentes, como Cientista de Dados com Bartender, vamos uma baixa similaridade, devido ao seu $ \theta $ ser próximo de 90, o que fará seu cosseno ser próximo de 0.

Aqui calculamos a similaridade do cosse entre um vetor TF-IDF do texto do usuário e cada vetor TF-IDF da vaga de emprego.

In [ ]:
todos_empregos.head()

,ID_emprego,texto,título
0,111,server tacolicious palo alto part time tacolic...,Server @ Tacolicious
1,113,kitchen staff chef claude lane san francisco p...,Kitchen Staff/Chef @ Claude Lane
2,117,bartender machka restaurants corp san francisc...,Bartender @ Machka Restaurants Corp.
3,121,server teriyaki house brisbane part time serve...,Server @ Teriyaki House
4,127,kitchen staff chef rosa mexicano sunset los an...,Kitchen Staff/Chef @ Rosa Mexicano - Sunset


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

tfidf_jobid = tfidf_vectorizer.fit_transform((todos_empregos['texto']))
tfidf_jobid

<84090x50754 sparse matrix of type '<class 'numpy.float64'>'
	with 8263698 stored elements in Compressed Sparse Row format>

In [ ]:
user_q

,ID_candidato,texto
186,326,java developer


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
user_tfidf = tfidf_vectorizer.transform(user_q['texto'])
cos_similarity_tfidf = map(lambda x: cosine_similarity(user_tfidf, x),tfidf_jobid)


In [ ]:
output2 = list(cos_similarity_tfidf)

###  Função para obter as recomendações principais por pontuação

- top será uma lista com as 10 vagas de emprego mais relevantes ao perfil do usuário.
- scores serão as pontuações de similaridade de cosseno associadas às 10 vagas de emprego mais relevantes.

In [ ]:
def obter_recomendação(top, df_all, scores):
  recomendação = pd.DataFrame(columns = ['ID_candidato', 'ID_emprego',  'título', 'pontuação'])
  count = 0
  for i in top:
      recomendação.at[count, 'ID_candidato'] = candidato
      recomendação.at[count, 'ID_emprego'] = df_all['ID_emprego'][i]
      recomendação.at[count, 'título'] = df_all['título'][i]
      recomendação.at[count, 'pontuação'] =  scores[count]
      count += 1
  return recomendação

##  As principais recomendações usando TF-IDF

Vamos criar uma lista com as 10 vagas mais relevantes para o usuário 326

In [ ]:
top = sorted(range(len(output2)), key=lambda i: output2[i], reverse=True)[:10]
top

[69346, 63958, 40385, 3231, 40634, 71496, 76180, 16225, 7108, 10301]

Obtemos o placar de similaridade de cada trabalho

In [ ]:
lista_pontuações = [output2[i][0][0] for i in top]
lista_pontuações

[0.7494775251900346,
 0.7408859264648848,
 0.7370067616428033,
 0.6716673937124862,
 0.6450373772174958,
 0.6255315736496132,
 0.5922913263911141,
 0.5302305793452344,
 0.5105340708444382,
 0.4867887337500865]

Obtemos essas pontuações em ordem decrescente

In [ ]:
obter_recomendação(top,todos_empregos, lista_pontuações)

,ID_candidato,ID_emprego,título,pontuação
0,326,303112,Java Developer @ TransHire,0.749478
1,326,294684,Java Developer @ Kavaliro,0.740886
2,326,269922,Entry Level Java Developer / Jr. Java Develope...,0.737007
3,326,141831,Lead Java/J2EE Developer - Contract to Hire @ ...,0.671667
4,326,270171,Senior Java Developer - Contract to Hire - Gre...,0.645037
5,326,305264,Sr. Java Developer @ Paladin Consulting Inc,0.625532
6,326,309945,"Java Software Engineer @ iTech Solutions, Inc.",0.592291
7,326,245753,Java Administrator @ ConsultNet,0.530231
8,326,146640,Jr. Java Developer @ Paladin Consulting Inc,0.510534
9,326,150882,Java Consultant - Mobile Apps Development @ Co...,0.486789


In [ ]:
applicant_ids = np.array([2, 3, 6,  11, 326])

In [ ]:
informacoes_candidatos = df_candidato_final[df_candidato_final['ID_candidato'].isin(applicant_ids)].drop_duplicates(subset='texto')
informacoes_candidatos

,ID_candidato,texto
0,2,writer uloop blog volunteer
1,3,market intern server prep cook
2,6,project assistant
4,11,cashier
186,326,java developer


In [ ]:
# Inicialize a lista para armazenar os DataFrames
df_candidatos_2_3_6_11_326 = []

for u in applicant_ids:
    index = np.where(df_candidato_final['ID_candidato'] == u)[0][0]
    user_q = df_candidato_final.iloc[[index]]
    user_tfidf = tfidf_vectorizer.transform(user_q['texto'])
    cos_similarity_tfidf = map(lambda x: cosine_similarity(user_tfidf, x), tfidf_jobid)
    output = list(cos_similarity_tfidf)
    top = sorted(range(len(output)), key=lambda i: output[i], reverse=True)[:10]
    lista_pontuações = [output[i][0][0] for i in top]
    # Use a função get_recommendation para obter as recomendações
    recommendations_df = obter_recomendação(top, todos_empregos, lista_pontuações)
    # Adicione o DataFrame à lista
    df_candidatos_2_3_6_11_326.append(recommendations_df)

In [ ]:
# Salvar random_user_dfs
with open('df_candidatos_2_3_6_11_326.pkl', 'wb') as file:
    pickle.dump(df_candidatos_2_3_6_11_326, file)

In [ ]:
# Caminho para o arquivo pickle no Google Drive
caminho_arquivo = '/content/drive/MyDrive/Dados_Empregos/df_candidatos_2_3_6_11_326.pkl'

# Carregue os dados do arquivo pickle
with open(caminho_arquivo, 'rb') as arquivo:
    df_candidatos_2_3_6_11_326 = pickle.load(arquivo)

In [ ]:
df_candidatos_2_3_6_11_326[-0]

,ID_candidato,ID_emprego,título,pontuação
0,326,269751,Specialized Writer @ OfficeTeam,0.463461
1,326,261122,Technical Writer @ Stivers Staffing Talent Net...,0.444053
2,326,266549,Procedure Writer- RN - Registered Nurse! @ Pro...,0.330006
3,326,263149,Copy Writer NEEDED ASAP! 9+ month Temp!!! @ Of...,0.319584
4,326,255776,Volunteer Manager @ VITAS Healthcare,0.317294
5,326,244541,"Hospice Volunteer Coordinator, HomeCare, Hartf...",0.316041
6,326,282989,Grants Coordinator/Writer @ Paul D. Camp Commu...,0.30472
7,326,293766,Volunteer Coordinator @ CHRISTUS Continuing Care,0.295702
8,326,293791,Hospice Volunteer Coordinator - Part Time - Ki...,0.288174
9,326,253866,Volunteer Specialist @ Franciscan Community Se...,0.288106


## As principais recomendações usando Count Vectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
count_vectorizer = CountVectorizer()

count_jobid = count_vectorizer.fit_transform((todos_empregos['texto'])) #fitting and transforming the vector
count_jobid

<84090x50754 sparse matrix of type '<class 'numpy.int64'>'
	with 8263698 stored elements in Compressed Sparse Row format>

In [ ]:
output3

[]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
user_count = count_vectorizer.transform(user_q['texto'])
cos_similarity_countv = map(lambda x: cosine_similarity(user_count, x),count_jobid)
output3 = list(cos_similarity_countv)

In [ ]:
top = sorted(range(len(output3)), key=lambda i: output3[i], reverse=True)[:10]
lista_pontos = [output3[i][0][0] for i in top]
obter_recomendação(top, todos_empregos, lista_pontos)

,ID_candidato,ID_emprego,título,pontuação


In [ ]:
# Inicializar a lista para armazenar os DataFrames
df2_candidatos_2_3_6_11_326 = []

# Loop para 5 usuários aleatórios
for user_id in applicant_ids:
    index = np.where(df_candidato_final['ID_candidato'] == user_id)[0][0]
    user_q = df_candidato_final.iloc[[index]]
    user_count = count_vectorizer.transform(user_q['texto'])

    cos_similarity_countv = map(lambda x: cosine_similarity(user_count, x), count_jobid)
    output = list(cos_similarity_countv)

    top = sorted(range(len(output)), key=lambda i: output[i][0][0], reverse=True)[:10]
    lista_pontos = [output[i][0][0] for i in top]

    # Aqui você precisará definir sua própria função get_recommendation
    user_df = obter_recomendação(top, todos_empregos, lista_pontos)

    # Adicionar o DataFrame do usuário à lista
    df2_candidatos_2_3_6_11_326.append(user_df)

In [ ]:
# Salvar random_user_dfs2
with open('df2_candidatos_2_3_6_11_326.pkl', 'wb') as file:
    pickle.dump(df2_candidatos_2_3_6_11_326, file)

In [ ]:
# Caminho para o arquivo pickle no Google Drive
caminho_arquivo2 = '/content/drive/MyDrive/Dados_Empregos/df2_candidatos_2_3_6_11_326.pkl'

# Carregue os dados do arquivo pickle
with open(caminho_arquivo2, 'rb') as arquivo:
    df2_candidatos_2_3_6_11_326 = pickle.load(arquivo)

In [ ]:
df2_candidatos_2_3_6_11_326[-1]

,ID_candidato,ID_emprego,título,pontuação
0,326,303112,Java Developer @ TransHire,0.635001
1,326,294684,Java Developer @ Kavaliro,0.600245
2,326,269922,Entry Level Java Developer / Jr. Java Develope...,0.571726
3,326,141831,Lead Java/J2EE Developer - Contract to Hire @ ...,0.496907
4,326,270171,Senior Java Developer - Contract to Hire - Gre...,0.481757
5,326,309945,"Java Software Engineer @ iTech Solutions, Inc.",0.454673
6,326,305264,Sr. Java Developer @ Paladin Consulting Inc,0.406017
7,326,245753,Java Administrator @ ConsultNet,0.378968
8,326,150882,Java Consultant - Mobile Apps Development @ Co...,0.363216
9,326,146640,Jr. Java Developer @ Paladin Consulting Inc,0.323381


## Resultados

- Java Developer

  - TF-IDF
    - Tem maiores scores, isso quer dizer que este algoritmo considera a frequência relativa das palavras
  - CountVectorizer
    - Menores scores